In [1]:
#imports
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table   
from astropy.coordinates import SkyCoord     
from astropy import units as u                   
from astropy.cosmology import Planck18 as cosmo
from astropy.table import unique
from astropy.visualization import make_lupton_rgb
from scipy.stats import binned_statistic
from kcorrect.kcorrect import Kcorrect
import pandas as pd


# 1. Data Loading & Quality Assurance
This section loads the raw DESI `.fits` catalog and applies baseline quality filters. We remove targets with hardware warnings (`ZWARN != 0`) and high redshift uncertainty to ensure our visual structures are based on high-confidence spectroscopic data.

In [2]:
#read in the catalog
path_to_catalog = '/users/PCON0003/ulricclaar/desi-openspace-visualization/data/zall-pix-iron.fits'  
catalog = Table.read(path_to_catalog)
print('Total number of objects:', len(catalog))

Total number of objects: 28425963


In [3]:
# Filter out entries with ZWARN != 0
if 'ZWARN' in catalog.colnames:
    catalog = catalog[catalog['ZWARN'] == 0]

#Filter out high ZERR values
if 'ZERR' in catalog.colnames:
    catalog = catalog[catalog['ZERR'] < 0.001]

# Delta Chi-Squared Cut (High Confidence Redshifts)
deltachi2_min = 40
catalog = catalog[catalog['DELTACHI2'] > deltachi2_min]

#Filter out duplicates
catalog = unique(catalog, keys=['TARGET_RA', 'TARGET_DEC'])
print('Objects kept:', len(catalog))

Objects kept: 18849581


# 2. Spatial Bounding & Flux Cleaning
Here we constrain the dataset to valid celestial coordinates and a specific cosmic depth (redshift 0.01 to 3.0) to remove local artifacts and extremely sparse deep-space objects. We also clip unphysical negative imaging fluxes caused by telescope noise.

In [4]:
# Trim catalog
RA_max = 360
RA_min = 0
DEC_max = 90
DEC_min = -90
catalog = catalog[(catalog['TARGET_RA'] < RA_max) & (catalog['TARGET_RA'] > RA_min) & (catalog['TARGET_DEC'] < DEC_max) & (catalog['TARGET_DEC'] > DEC_min)]
print('Objects kept:', len(catalog))
# Limit redshift
z_min = 0.01 # Below this, they're mostly stars or errors
z_max = 3    # DESI goes higher than this, but it is sparse and harder to visualize
catalog = catalog[(catalog['Z'] > z_min) & (catalog['Z'] < z_max)]
print('Objects kept:', len(catalog))
# Remove galaxies with unrealisting imaging fluxes
flux_g_max = 50  # These are quite faint. You calso can't have negative flux!
flux_g_min = 0
flux_r_max = 50  
flux_r_min = 0
flux_z_max = 50  
flux_z_min = 0
flux_W1_max = 50  
#For all flux W1 and W2 we clip the values to a minimum of 0.001 to avoid negative fluxes, which can occur due to noise in the measurements. This ensures that we only keep physically meaningful flux values for our analysis.
safe_flux_W1 = np.clip(catalog['FLUX_W1'], 0.001, None)
flux_W2_max = 50 
safe_flux_W2 = np.clip(catalog['FLUX_W2'], 0.001, None)

catalog = catalog[(catalog['FLUX_G'] < flux_g_max) & (catalog['FLUX_G'] > flux_g_min) & (catalog['FLUX_R'] < flux_r_max) & (catalog['FLUX_R'] > flux_r_min) & (catalog['FLUX_Z'] < flux_z_max) & (catalog['FLUX_Z'] > flux_z_min) & (catalog['FLUX_W1'] < flux_W1_max) & (catalog['FLUX_W2'] < flux_W2_max)]


print('Objects kept:', len(catalog))

Objects kept: 18849581
Objects kept: 14344852
Objects kept: 10053068


# 3. 3D Cartesian Coordinate Projection
Because 3D rendering engines use flat Euclidean space, we use Astropy's `Planck18` cosmology model. This calculates the comoving distance to each galaxy based on its redshift, allowing us to project spherical sky coordinates (RA, Dec) into standard 3D Cartesian vectors (x, y, z) in megaparsecs.

In [5]:
comoving_distance = cosmo.comoving_distance(catalog['Z'])  # Calculate the comoving distance to each object based on its redshift and the cosmology
coords = SkyCoord(ra=catalog['TARGET_RA'].value * u.degree, dec=catalog['TARGET_DEC'].value * u.degree, distance=comoving_distance) 
print('Objects kept:', len(catalog)) # Astropy converts this into a 3D coordinate system

Objects kept: 10053068


In [6]:
#Define 3d coords
x = coords.cartesian.x  # These are still astropy objects, so they have assigned units (Mpc)
y = coords.cartesian.y
z = coords.cartesian.z
print('Objects kept:', len(catalog))

Objects kept: 10053068


In [7]:
#Define redshift and fluxes
redshift = catalog['Z'] 
flux_g = catalog['FLUX_G']
flux_r = catalog['FLUX_R']
flux_z = catalog['FLUX_Z']
flux_w1 = catalog['FLUX_W1']
flux_w2 = catalog['FLUX_W2']
#xtra
object_type = catalog['SPECTYPE']
zerr = catalog['ZERR']
zwarn = catalog['ZWARN']
print('Objects kept:', len(catalog))

Objects kept: 10053068


# 4. Photometry & Rest-Frame Corrections
This block handles the physics of galaxy brightness. We convert raw telescope fluxes into apparent magnitudes to prepare for intrinsic luminosity and color derivations.

In [8]:
#Define apparent magnitudes
app_mag_G = -2.5*np.log10(catalog['FLUX_G'])+22.5 
app_mag_R = -2.5*np.log10(catalog['FLUX_R'])+22.5 
app_mag_Z = -2.5*np.log10(catalog['FLUX_Z'])+22.5 
#app_mag_W1 = -2.5*np.log10(catalog['FLUX_W1'])+22.5
#app_mag_W2 = -2.5*np.log10(catalog['FLUX_W2'])+22.5
app_mag_W1 = -2.5 * np.log10(np.clip(catalog['FLUX_W1'], 0.001, None)) + 22.5
app_mag_W2 = -2.5 * np.log10(np.clip(catalog['FLUX_W2'], 0.001, None)) + 22.5
print('Objects kept:', len(catalog))

Objects kept: 10053068


In [9]:
#Define absolute magnitudes
def apparent_to_absolute_magnitude(m, z):
    d_L = cosmo.luminosity_distance(z)  
    M = m - 5 * np.log10(d_L.to(u.pc).value) + 5  
    return M
abs_mag_g = apparent_to_absolute_magnitude(app_mag_G, redshift)
abs_mag_r = apparent_to_absolute_magnitude(app_mag_R, redshift)
abs_mag_z = apparent_to_absolute_magnitude(app_mag_Z, redshift)
abs_mag_w1 = apparent_to_absolute_magnitude(app_mag_W1, redshift)
abs_mag_w2 = apparent_to_absolute_magnitude(app_mag_W2, redshift)
print('Objects kept:', len(catalog))

Objects kept: 10053068


In [10]:
# 1. Filter existing 'catalog' for valid galaxies
valid = (catalog['SPECTYPE'] == 'GALAXY') & (catalog['ZWARN'] == 0) & (catalog['Z'] > 0)
galaxies = catalog[valid]

redshifts = galaxies['Z']
print('Step 1 completed: Filtered galaxies and extracted redshifts.')
# 2. Extract and convert photometry (nanomaggies to maggies)
maggies = np.column_stack([
    galaxies['FLUX_G'] * 1e-9,
    galaxies['FLUX_R'] * 1e-9,
    galaxies['FLUX_Z'] * 1e-9
])

# Convert inverse variances (scale by 1e18)
maggies_ivar = np.column_stack([
    galaxies['FLUX_IVAR_G'] * 1e18,
    galaxies['FLUX_IVAR_R'] * 1e18,
    galaxies['FLUX_IVAR_Z'] * 1e18
])
print('Step 2 completed: Extracted and converted photometry to maggies and inverse variances.')
# 3. Initialize the Kcorrect object and fit the templates
kc = Kcorrect(responses=['decam_g', 'decam_r', 'decam_z'])
coeffs = kc.fit_coeffs(redshift=redshifts, maggies=maggies, ivar=maggies_ivar)
print('Step 3 completed: Initialized Kcorrect and fitted templates to obtain coefficients.')
# 4. Calculate Absolute Magnitudes natively
# In kcorrect v5, this one line handles the K-correction, the distance modulus, 
# and the Planck18 cosmology calculations all at once
absolute_mags = kc.absmag(redshift=redshifts, maggies=maggies, ivar=maggies_ivar, coeffs=coeffs)
print('Step 4 completed: Calculated absolute magnitudes using Kcorrect.')
# 5. Extract r-band (index 1) and z-band (index 2)
galaxies['ABS_MAG_R'] = absolute_mags[:, 1]
galaxies['ABS_MAG_Z'] = absolute_mags[:, 2]
print('Step 5 completed: Extracted r-band and z-band absolute magnitudes.')
# 6. Calculate the rest-frame color
galaxies['REST_COLOR_RZ'] = galaxies['ABS_MAG_R'] - galaxies['ABS_MAG_Z']
print('Step 6 completed: Calculated rest-frame color (r - z) for galaxies.')

Step 1 completed: Filtered galaxies and extracted redshifts.
Step 2 completed: Extracted and converted photometry to maggies and inverse variances.
Step 3 completed: Initialized Kcorrect and fitted templates to obtain coefficients.
Step 4 completed: Calculated absolute magnitudes using Kcorrect.
Step 5 completed: Extracted r-band and z-band absolute magnitudes.
Step 6 completed: Calculated rest-frame color (r - z) for galaxies.


# 5. Color Mapping & OpenSpace Export
Here we split the data: Quasars get a power-law K-correction, while Galaxies run through spectral template fitting (`kcorrect`). Using the Lupton algorithm, we map our corrected physical luminosities into an RGB color space. Finally, the split datasets are stitched back together and exported as a clean `.csv` ready for OpenSpace.

In [11]:
# ---------------------------------------------------------
# Helper Function: Custom RGB Generator (g-r mapping)
# ---------------------------------------------------------
def get_boosted_rgb(mag_r, mag_g):
    """Applies custom luminosity conversion and blue-boosted Lupton mapping."""
    # Suppress warnings for negative fluxes inside logs if any exist
    with np.errstate(invalid='ignore', divide='ignore'):
        lum_r = 10 ** (-0.4 * mag_r)
        lum_g = 10 ** (-0.4 * mag_g)

    norm = np.nanpercentile(lum_r, 5)
    lum_r = lum_r / norm
    lum_g = lum_g / norm

    rest_R = lum_r
    rest_G = (lum_r + lum_g) / 2.0
    rest_B = lum_g * 1.5 

    rgb_array = make_lupton_rgb(rest_R, rest_G, rest_B, stretch=0.2, Q=2)
    return rgb_array / 255.0

# =========================================================
# 1. PROCESS QUASARS (Power-Law Method)
# =========================================================
print("Processing Quasars...")
valid_qso = (catalog['SPECTYPE'] == 'QSO') & (catalog['ZWARN'] == 0) & (catalog['Z'] > 0)
qsos = catalog[valid_qso]
z_qso = qsos['Z']

flux_g_qso = qsos['FLUX_G'] * 1e-9
flux_r_qso = qsos['FLUX_R'] * 1e-9
flux_z_qso = qsos['FLUX_Z'] * 1e-9

app_mag_g_qso = -2.5 * np.log10(flux_g_qso)
app_mag_r_qso = -2.5 * np.log10(flux_r_qso)
app_mag_z_qso = -2.5 * np.log10(flux_z_qso)
app_color_rz_qso = app_mag_r_qso - app_mag_z_qso

# Quasar K-Correction
alpha = -0.5
k_corr_qso = -2.5 * (1 + alpha) * np.log10(1 + z_qso)
dl_pc_qso = cosmo.luminosity_distance(z_qso).value * 1e6
dm_qso = 5.0 * np.log10(dl_pc_qso / 10.0)

abs_mag_g_qso = app_mag_g_qso - dm_qso - k_corr_qso
abs_mag_r_qso = app_mag_r_qso - dm_qso - k_corr_qso
abs_mag_z_qso = app_mag_z_qso - dm_qso - k_corr_qso
rest_color_rz_qso = abs_mag_r_qso - abs_mag_z_qso

# Generate Quasar RGBs
rgb_uncorr_qso = get_boosted_rgb(app_mag_r_qso, app_mag_g_qso).reshape(-1, 3)
rgb_corr_qso = get_boosted_rgb(abs_mag_r_qso, abs_mag_g_qso).reshape(-1, 3)

# =========================================================
# 2. PROCESS GALAXIES (kcorrect Template Method)
# =========================================================
print("Processing Galaxies (Fitting templates, this may take a minute)...")
valid_gal = (catalog['SPECTYPE'] == 'GALAXY') & (catalog['ZWARN'] == 0) & (catalog['Z'] > 0)
gals = catalog[valid_gal]
z_gal = gals['Z']

flux_g_gal = gals['FLUX_G'] * 1e-9
flux_r_gal = gals['FLUX_R'] * 1e-9
flux_z_gal = gals['FLUX_Z'] * 1e-9

app_mag_g_gal = -2.5 * np.log10(flux_g_gal)
app_mag_r_gal = -2.5 * np.log10(flux_r_gal)
app_mag_z_gal = -2.5 * np.log10(flux_z_gal)
app_color_rz_gal = app_mag_r_gal - app_mag_z_gal

# Prepare arrays for kcorrect
maggies_gal = np.column_stack([flux_g_gal, flux_r_gal, flux_z_gal])
ivar_gal = np.column_stack([
    gals['FLUX_IVAR_G'] * 1e18,
    gals['FLUX_IVAR_R'] * 1e18,
    gals['FLUX_IVAR_Z'] * 1e18
])

# Run kcorrect
kc = Kcorrect(responses=['decam_g', 'decam_r', 'decam_z'])
coeffs_gal = kc.fit_coeffs(redshift=z_gal, maggies=maggies_gal, ivar=ivar_gal)
abs_mags_gal = kc.absmag(redshift=z_gal, maggies=maggies_gal, ivar=ivar_gal, coeffs=coeffs_gal)

# Extract galaxy absolute magnitudes (index 0=g, 1=r, 2=z based on our filter list)
abs_mag_g_gal = abs_mags_gal[:, 0]
abs_mag_r_gal = abs_mags_gal[:, 1]
abs_mag_z_gal = abs_mags_gal[:, 2]
rest_color_rz_gal = abs_mag_r_gal - abs_mag_z_gal

# Generate Galaxy RGBs
rgb_uncorr_gal = get_boosted_rgb(app_mag_r_gal, app_mag_g_gal).reshape(-1, 3)
rgb_corr_gal = get_boosted_rgb(abs_mag_r_gal, abs_mag_g_gal).reshape(-1, 3)



Processing Quasars...


Processing Galaxies (Fitting templates, this may take a minute)...


/tmp/ipykernel_1939971/1356665212.py:8: RuntimeWarning: overflow encountered in power
  lum_r = 10 ** (-0.4 * mag_r)
/tmp/ipykernel_1939971/1356665212.py:9: RuntimeWarning: overflow encountered in power
  lum_g = 10 ** (-0.4 * mag_g)


In [12]:
print("Stitching arrays back into master catalog...")

# 1. Initialize the new columns with NaNs (Not a Number) to hold our data safely
catalog['APP_MAG_R'] = np.nan
catalog['ABS_MAG_R'] = np.nan
catalog['REST_COLOR_RZ'] = np.nan
catalog['color_r'] = np.nan
catalog['color_g'] = np.nan
catalog['color_b'] = np.nan

# 2. Insert the Galaxy data using the valid_gal mask
catalog['APP_MAG_R'][valid_gal] = app_mag_r_gal
catalog['ABS_MAG_R'][valid_gal] = abs_mag_r_gal
catalog['REST_COLOR_RZ'][valid_gal] = rest_color_rz_gal
catalog['color_r'][valid_gal] = rgb_corr_gal[:, 0]
catalog['color_g'][valid_gal] = rgb_corr_gal[:, 1]
catalog['color_b'][valid_gal] = rgb_corr_gal[:, 2]

# 3. Insert the Quasar data using the valid_qso mask
catalog['APP_MAG_R'][valid_qso] = app_mag_r_qso
catalog['ABS_MAG_R'][valid_qso] = abs_mag_r_qso
catalog['REST_COLOR_RZ'][valid_qso] = rest_color_rz_qso
catalog['color_r'][valid_qso] = rgb_corr_qso[:, 0]
catalog['color_g'][valid_qso] = rgb_corr_qso[:, 1]
catalog['color_b'][valid_qso] = rgb_corr_qso[:, 2]

# 4. Drop anything we DIDN'T process 
# (like stars or bad data) so they don't break the OpenSpace size math
valid_master_mask = valid_gal | valid_qso
catalog = catalog[valid_master_mask]

print(f"Merge complete. Master catalog ready for OpenSpace with {len(catalog)} objects.")

Stitching arrays back into master catalog...
Merge complete. Master catalog ready for OpenSpace with 10053068 objects.


In [22]:
# 1. Isolate your absolute magnitudes 
mags = catalog['ABS_MAG_R']

# 2. Set the visual boundaries for OpenSpace (min size and max size)
scale_min = 0.1
scale_max = 3.0

# 3. Find the brightest and dimmest bounds in your dataset
mag_min = np.nanmin(mags)  # Most negative (brightest)
mag_max = np.nanmax(mags)  # Least negative (dimmest)

# 4. Mathematically invert and normalize the scale
# This formula makes sure the brightest galaxies map to 3.0 and the dimmest map to 0.1
normalized_scale = (mags - mag_max) / (mag_min - mag_max)
catalog['size_scale'] = scale_min + (normalized_scale * (scale_max - scale_min))

In [ ]:
# ==========================================
# HYBRID GALAXY CLASSIFICATION PIPELINE
# ==========================================

# 1. Initialize object subtype column to 0 (default/unknown)
catalog['object_subtype'] = 0

# Set the redshift cutoff where image resolution breaks down
Z_CUTOFF = 0.5

# Clean the MORPHTYPE strings (handles trailing whitespace or byte encodings)
morphtype = np.char.strip(catalog['MORPHTYPE'].astype(str))
z = catalog['Z']

# ------------------------------------------
# ZONE A: NEARBY UNIVERSE (z < 0.5) -> Visual Tractor Data
# ------------------------------------------
is_nearby = z < Z_CUTOFF

catalog['object_subtype'][is_nearby & (morphtype == 'EXP')] = 1   # Spiral (Exponential disk)
catalog['object_subtype'][is_nearby & (morphtype == 'DEV')] = 2   # Elliptical (de Vaucouleurs)
catalog['object_subtype'][is_nearby & (morphtype == 'PSF')] = 3   # Quasar/Star (Point Source)
catalog['object_subtype'][is_nearby & (morphtype == 'SER')] = 4  # Lenticular (Sersic profile)
catalog['object_subtype'][is_nearby & (morphtype == 'REX')] = 5   # Irregular/Round Exponential

# ------------------------------------------
# ZONE B: DEEP UNIVERSE (z >= 0.5) -> Spectroscopic & Color Data
# ------------------------------------------
is_distant = z >= Z_CUTOFF

# DESI Dark Time Bitmask definitions
LRG_BIT = 1  # Luminous Red Galaxy
ELG_BIT = 2  # Emission Line Galaxy
QSO_BIT = 4  # Quasar

is_lrg = (catalog['DESI_TARGET'] & LRG_BIT) != 0
is_elg = (catalog['DESI_TARGET'] & ELG_BIT) != 0
is_qso = (catalog['DESI_TARGET'] & QSO_BIT) != 0

catalog['object_subtype'][is_distant & is_elg] = 1   # Distant star-forming -> Spiral
catalog['object_subtype'][is_distant & is_lrg] = 2   # Distant quiescent -> Elliptical
catalog['object_subtype'][is_distant & is_qso] = 3   # Distant AGN -> Quasar

# ------------------------------------------
# FALLBACK: Catch unclassified objects (BGS / edge cases)
# ------------------------------------------
unassigned = catalog['object_subtype'] == 0

# If unclassified, use rest-frame color split
is_red = catalog['color_r'] > catalog['color_b']
is_blue = catalog['color_b'] >= catalog['color_r']

catalog['object_subtype'][unassigned & is_blue] = 1  # Unclassified Blue -> Spiral
catalog['object_subtype'][unassigned & is_red] = 2   # Unclassified Red -> Elliptical

# Section 6: Color vs. Morphology Scientific Audit

Before exporting our finalized coordinate skeleton to a CSV for OpenSpace ingestion, we conduct an automated statistical audit to verify the physical consistency between our photometric colors and structural imaging morphology.

### Structural Verification Pipeline
We cross-reference our spectral color classifications against structural profile fits from imaging data:
* **Spiral Disk Models (`EXP`):** Checks the match rate of galaxies best fit by an exponential disk light profile against blue, star-forming photometric subtypes.
* **Elliptical Bulge Models (`DEV`):** Checks the match rate of galaxies best fit by a de Vaucouleurs ($r^{1/4}$) profile against red, passive elliptical subtypes (`Subtype 2`).

This final verification prints an accuracy scorecard (`spiral_accuracy` and `elliptical_accuracy`), validating the scientific integrity of our 3D cosmic web before rendering.

In [24]:
# ASSEMBLE AND EXPORT THE FINAL MASTER CSV FILE
final_production_table = Table(
    [
        x, y, z, catalog['color_r'], catalog['color_g'], catalog['color_b'],
        catalog['TARGET_RA'], catalog['TARGET_DEC'], catalog['Z'], catalog['size_scale'], catalog['object_subtype']
    ], 
    names=[
        'x', 'y', 'z', 'color_r', 'color_g', 'color_b',
        'TARGET_RA', 'TARGET_DEC', 'Z', 'size_scale', 'object_subtype'
    ]
)

# Clean up any 2D telescope column array dimensions
for col in final_production_table.colnames:
    if len(final_production_table[col].shape) > 1:
        final_production_table[col] = np.squeeze(final_production_table[col])

# Overwrite the production CSV file in the assets folder
final_production_table.write('/users/PCON0003/ulricclaar/desi-openspace-visualization/data/desi_catalog_fullsurvey_test.csv', format='csv', overwrite=True)
print("SUCCESS: The master file is completely saved and ready for OpenSpace.")
print('Objects kept:', len(catalog))

SUCCESS: The master file is completely saved and ready for OpenSpace.
Objects kept: 10053068


In [26]:
# Path to your test CSV
test_csv_path = '/users/PCON0003/ulricclaar/desi-openspace-visualization/data/desi_catalog_fullsurvey_test.csv'

print("Loading test catalog for inspection...")
# Load only the first 5 rows to save memory and time
df_test = pd.read_csv(test_csv_path, nrows=5)

# The master list of columns OpenSpace and your slicer need
expected_columns = [
    'x', 'y', 'z', 
    'color_r', 'color_g', 'color_b', 
    'TARGET_RA', 'TARGET_DEC', 'Z', 
    'size_scale', 'object_subtype'
]

# Check for missing columns
missing_cols = [col for col in expected_columns if col not in df_test.columns]

if missing_cols:
    print(f"❌ WARNING: Missing columns detected: {missing_cols}")
else:
    print("✅ SUCCESS: All expected columns are present!")

print("\n--- First 5 Rows Preview ---")
# This prints the actual data so you can verify size_scale and object_subtype aren't just zeroes or NaNs
print(df_test[expected_columns].head())

Loading test catalog for inspection...
✅ SUCCESS: All expected columns are present!

--- First 5 Rows Preview ---
             x         y         z   color_r   color_g   color_b  TARGET_RA  \
0  3290.413275  0.003913  0.963933  0.823529  0.745098  1.000000   0.000068   
1  2966.791999  0.008269  0.839490  1.000000  0.827451  0.988235   0.000160   
2  2980.762302  0.009792  0.842655  0.815686  0.741176  1.000000   0.000188   
3  3839.762730  0.014878  1.191451  0.796078  0.729412  1.000000   0.000222   
4  1136.112280  0.004645  0.274680  0.996078  0.749020  0.749020   0.000234   

   TARGET_DEC         Z  size_scale  object_subtype  
0    5.392670  0.963933    0.104216               1  
1    4.778598  0.839490    0.104163               1  
2    3.324588  0.842655    0.104414               1  
3    3.247259  1.191451    0.104536               1  
4   -0.884509  0.274680    0.104174               5  
